# FairPriceDemo

Questa demo carica i modelli reali (LightGBM, Isolation Forest) e implementa **l'intera pipeline di anomaly detection** identica a quella addestrata (con TreeSHAP, regole di coerenza deterministiche e Z-Score), producendo l'output JSON completo.

In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import joblib
import shap
import lightgbm as lgb
from sklearn.ensemble import IsolationForest

warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 100)

DATA_FOLDER = "data"
MODEL_DIR = DATA_FOLDER + "/models"


In [2]:
# 1. Caricamento Dataset e Modelli di Prezzo
print("Caricamento Dataset e Modelli di Prezzo...")
try:
    train_df = pd.read_csv(f"{DATA_FOLDER}/train_with_price.csv")
    test_df = pd.read_csv(f"{DATA_FOLDER}/test_with_price.csv")

    price_model = joblib.load(f'{MODEL_DIR}/lgbm_price_model.joblib')
    model_q10 = joblib.load(f'{MODEL_DIR}/lgbm_price_q10.joblib')
    model_q90 = joblib.load(f'{MODEL_DIR}/lgbm_price_q90.joblib')

    with open(f'{MODEL_DIR}/price_model_config.json', 'r') as f:
        price_config = json.load(f)
        PRICE_FEATURES = price_config['features']
        PRICE_CATEGORICAL_COLS = price_config['categorical_cols']
        PRICE_NUMERICAL_COLS = price_config['numerical_cols']
        residual_thresholds = list(price_config['residual_thresholds'].values())
except Exception as e:
    print(f"Errore durante il caricamento: {e}\nAssicurati di aver eseguito il notebook principale.")


Caricamento Dataset e Modelli di Prezzo...


In [3]:
# 2. Inizializzazione Configurazione Anomalie
REFERENCE_YEAR = 2025

ANOMALY_EXCLUDED_COLS = ["price_negotiable", "price_tax_deductible", "seller_is_dealer", "n_missing_fields"]
ORDINAL_COLS = [col for col in ["fuel_category", "transmission", "body_type"] if col not in ANOMALY_EXCLUDED_COLS]
FREQ_COLS = [col for col in ["make", "model"] if col not in ANOMALY_EXCLUDED_COLS]
NUMERICAL_ANOMALY_COLS = [col for col in PRICE_NUMERICAL_COLS + ["price", "price_per_year", "hp_per_cc"] if col not in ANOMALY_EXCLUDED_COLS]

# Feature derivate per l'anomaly detection (non presenti in train_with_price.csv)
train_df["car_age"] = (REFERENCE_YEAR - train_df["registration_year"]).clip(lower=0)
train_df["km_per_year"] = train_df["mileage_km"] / np.maximum(train_df["car_age"], 1)
train_df["price_per_year"] = train_df["price"] / np.maximum(train_df["car_age"], 1)
vol = train_df["cylinders_volume_cc"].replace(0, np.nan)
train_df["hp_per_cc"] = (train_df["power_hp"] / vol).fillna(0)

print("Inizializzazione Modelli di Anomaly Detection (Isolation Forest & SHAP)...")
# Carichiamo l'Isolation Forest salvata dal notebook principale.
# Se il file non esiste, la riaddestriamo con gli stessi parametri.
try:
    iso_forest = joblib.load(f'{MODEL_DIR}/iso_forest.joblib')
    explainer = shap.TreeExplainer(iso_forest)
    print("Modelli di anomalia caricati.")
except FileNotFoundError:
    print("Addestramento rapido Isolation Forest...")
    X_train_if = train_df[NUMERICAL_ANOMALY_COLS].copy()
    for col in ORDINAL_COLS:
        X_train_if[col] = train_df[col].astype("category").cat.codes
    for col in FREQ_COLS:
        freq = train_df[col].value_counts(normalize=True)
        X_train_if[f"{col}_freq"] = train_df[col].map(freq).fillna(0)
    X_train_if = X_train_if.fillna(0)

    iso_forest = IsolationForest(n_estimators=300, random_state=1234, n_jobs=-1)
    iso_forest.fit(X_train_if)
    explainer = shap.TreeExplainer(iso_forest)
    print("Modelli generati con successo.")

Inizializzazione Modelli di Anomaly Detection (Isolation Forest & SHAP)...
Modelli di anomalia caricati.


In [4]:
# 3. Pre-calcolo Thresholds, Mediane Comparabili e Peers
# 3a. Risk Thresholds (ricavati dai residual negativi di X_train_if)
if 'X_train_if' not in locals():
    X_train_if = train_df[NUMERICAL_ANOMALY_COLS].copy()
    for col in ORDINAL_COLS:
        X_train_if[col] = train_df[col].astype("category").cat.codes
    for col in FREQ_COLS:
        freq = train_df[col].value_counts(normalize=True)
        X_train_if[f"{col}_freq"] = train_df[col].map(freq).fillna(0)
    X_train_if = X_train_if.fillna(0)

train_scores = iso_forest.decision_function(X_train_if)
min_score = np.min(train_scores)
high_threshold = min_score * 0.66
medium_threshold = min_score * 0.33
low_threshold = 0.0

def assign_risk_category(scores, t_high, t_med, t_low):
    conditions = [
        scores < t_high,
        (scores >= t_high) & (scores < t_med),
        (scores >= t_med) & (scores < t_low),
        scores >= t_low
    ]
    return np.select(conditions, ["high", "medium", "low", "none"], default="none")

# 3b. Mediane comparabili per spiegazione SHAP
if "age_group" not in train_df.columns:
    train_df["car_age"] = (REFERENCE_YEAR - train_df["registration_year"]).clip(lower=0)
    train_df["age_group"] = pd.cut(train_df["car_age"], bins=[-1, 3, 7, 12, 100], labels=["0-3", "4-7", "8-12", "12+"])
medians_df = train_df.groupby(["make", "model", "age_group"], observed=False)[NUMERICAL_ANOMALY_COLS].median()

# 3c. Peer stats per check Z-Score 
def robust_mad(x):
    mad = np.median(np.abs(x - np.median(x)))
    return mad * 1.4826 if mad > 0 else 1.0

peer_stats = train_df.groupby(["make", "model", "registration_year"]).agg(
    peer_price_median=("price", "median"),
    peer_price_mad=("price", robust_mad)
).reset_index()


In [5]:
# 4. Pipeline Predittiva Prezzo
def prepare_lgbm_features(train_data, test_data, feature_cols, cat_cols):
    X_te = test_data[feature_cols].copy()
    for col in cat_cols:
        if col in train_data.columns:
            X_te[col] = pd.Categorical(X_te[col], categories=train_data[col].astype("category").cat.categories)
    return None, X_te

def assign_price_categories(residuals, thresholds, labels=["well_below_market", "below_market", "in_line", "above_market", "well_above_market"]):
    bins = [-np.inf, *thresholds, np.inf]
    return pd.cut(residuals, bins=bins, labels=labels)

def predict_random_ad(ad_df):
    _, X_ad = prepare_lgbm_features(train_df, ad_df, PRICE_FEATURES, PRICE_CATEGORICAL_COLS)
    ideal_price = float(np.exp(price_model.predict(X_ad)[0]))
    lower_bound = float(np.exp(model_q10.predict(X_ad)[0]))
    upper_bound = float(np.exp(model_q90.predict(X_ad)[0]))

    result = {
        "ideal_price_eur": ideal_price,
        "lower_bound_p10_eur": lower_bound,
        "upper_bound_p90_eur": upper_bound,
    }
    if "price" in ad_df.columns:
        residual = float(np.log(ad_df["price"].iloc[0]) - np.log(ideal_price))
        category = assign_price_categories(np.array([residual]), residual_thresholds)[0]
        result.update({     # type: ignore
            "asking_price_eur": float(ad_df["price"].iloc[0]),
            "log_residual": residual,
            "price_category": str(category),
        })
    return result


In [6]:
# 5. PIPELINE REALE DELLE ANOMALIE (Completa)
def check_coherence_rules(row):
    signals = []
    if row.get("is_new") and row.get("mileage_km", 0) > 1000:
        signals.append("is_new with high mileage")
    if row.get("is_new") and row.get("is_used"):
        signals.append("is_new and is_used both true")
    if pd.notna(row.get("power_kw")) and pd.notna(row.get("power_hp")):
        expected_hp = row.get("power_kw") * 1.35962
        if abs(row.get("power_hp") - expected_hp) > 5:
            signals.append("power_hp inconsistent with power_kw")
    fuel = str(row.get("fuel_category")).lower()
    if fuel == "electric" and (row.get("cylinders", 0) > 0 or row.get("cylinders_volume_cc", 0) > 0):
        signals.append("electric car with displacement or cylinders")
    if row.get("registration_year", 0) > REFERENCE_YEAR:
        signals.append("registration in the future")
    if row.get("is_new") and row.get("nr_prev_owners", 0) > 0:
        signals.append("is_new with previous owners")
    return "; ".join(signals) if signals else "None"

def test_anomaly_pipeline(ad_df):
    if "price" not in ad_df.columns:
        return {"error": "Anomaly detection requires an asking price."}
        
    ad_eval = ad_df.copy().reset_index(drop=True)
    
    # Feature Engineering per Anomalie
    ad_eval["car_age"] = (REFERENCE_YEAR - ad_eval["registration_year"]).clip(lower=0)
    ad_eval["km_per_year"] = ad_eval["mileage_km"] / np.maximum(ad_eval["car_age"], 1)
    ad_eval["price_per_year"] = ad_eval["price"] / np.maximum(ad_eval["car_age"], 1)
    
    vol = ad_eval.get("cylinders_volume_cc", 0)
    if isinstance(vol, pd.Series): vol = vol.iloc[0]
    
    if pd.isna(vol) or vol == 0:
        ad_eval["hp_per_cc"] = 0
    else:
        ad_eval["hp_per_cc"] = ad_eval["power_hp"] / vol
        
    # Encoding Dinamico (allineato al train)
    X_ad_if = pd.DataFrame(index=[0], columns=NUMERICAL_ANOMALY_COLS)
    for col in NUMERICAL_ANOMALY_COLS:
        X_ad_if[col] = ad_eval.get(col, 0)
            
    for col in ORDINAL_COLS:
        cat_type = train_df[col].astype("category")
        X_ad_if[col] = pd.Categorical(ad_eval[col], categories=cat_type.cat.categories).codes
        
    for col in FREQ_COLS:
        freq = train_df[col].value_counts(normalize=True)
        X_ad_if[f"{col}_freq"] = ad_eval[col].map(freq).fillna(0)
        
    X_ad_if = X_ad_if.fillna(0)
    
    # Isolation Forest & Risk Scoring
    score = iso_forest.decision_function(X_ad_if)[0]
    risk = assign_risk_category(np.array([score]), high_threshold, medium_threshold, low_threshold)[0]
    
    # TreeSHAP Explainability
    top_features_str = "None"
    if risk in ["high", "medium", "low"]:
        shap_vals = explainer.shap_values(X_ad_if)[0]
        top_3_indices = np.argsort(shap_vals)[:3] 
        
        make = str(ad_eval["make"].iloc[0]).strip().lower()
        model = str(ad_eval["model"].iloc[0]).strip().lower()
        age = ad_eval["car_age"].iloc[0]
        
        if age <= 3: age_grp = "0-3"
        elif age <= 7: age_grp = "4-7"
        elif age <= 12: age_grp = "8-12"
        else: age_grp = "12+"
        
        try:
            comparables_median = medians_df.loc[(make, model, age_grp)]     # type: ignore
        except KeyError:
            comparables_median = None
            
        top_features = []
        for idx in top_3_indices:
            feat_name = X_ad_if.columns[idx]
            feat_val = X_ad_if.iloc[0, idx]
            med_str = ""
            if comparables_median is not None and feat_name in comparables_median and pd.notna(comparables_median[feat_name]):
                med_str = f", mediana comparabili = {comparables_median[feat_name]:.2f}"
            top_features.append(f"{feat_name} = {feat_val:.2f}{med_str} [SHAP: {shap_vals[idx]:.2f}]")
        top_features_str = " | ".join(top_features)
        
    # Deterministic Coherence Rules
    coherence = check_coherence_rules(ad_eval.iloc[0])
    
    # Robust Z-Score contro i peers
    make_val, model_val, year_val = ad_eval["make"].iloc[0], ad_eval["model"].iloc[0], ad_eval["registration_year"].iloc[0]
    peer_match = peer_stats[(peer_stats["make"] == make_val) & 
                            (peer_stats["model"] == model_val) & 
                            (peer_stats["registration_year"] == year_val)]    

    z_score_signal = "None"
    if not peer_match.empty:
        med = peer_match["peer_price_median"].iloc[0]
        mad = peer_match["peer_price_mad"].iloc[0]
        z_score = (ad_eval["price"].iloc[0] - med) / mad
        z_score_signal = "True" if abs(z_score) > 3 else "False"

    return {
        "asking_price_eur": ad_eval["price"].iloc[0],
        "anomaly_score": round(score, 4),
        "risk_category": risk.upper(),
        "top_3_anomalous_features": top_features_str,
        "coherence_signals": coherence,
        "z_score_signal": z_score_signal
    }


In [7]:
# 6. Aggregazione Valutazione
def evaluate_listing(ad_df, return_json=False):
    price_results = predict_random_ad(ad_df)
    anomaly_results = test_anomaly_pipeline(ad_df)
    
    combined_result = {
        "asking_price": price_results.get("asking_price_eur"),
        "ideal_price": round(price_results.get("ideal_price_eur", 0.0), 2),
        "price_category": price_results.get("price_category"),
        "anomaly_score": anomaly_results.get("anomaly_score"),
        "risk_category": anomaly_results.get("risk_category"),
        "top_3_anomalous_features": anomaly_results.get("top_3_anomalous_features")
    }
    
    if return_json:
        return json.dumps(combined_result, indent=4)
        
    return combined_result

if __name__ == "__main__":
    # Estraiamo un annuncio a caso per il test
    sample_ad = test_df.sample(n=1, random_state=42).copy()
    display(sample_ad)
    
    evaluation_json = evaluate_listing(sample_ad, return_json=True)
    print("\n=== OUTPUT JSON DELLA VALUTAZIONE COMPLETA E REALE ===")
    print(evaluation_json)

,original_row,price,price_tax_deductible,price_negotiable,make,model,mileage_km,vehicle_type,body_type,nr_seats,nr_doors,body_color,paint_type,upholstery,upholstery_color,power_kw,power_hp,transmission,gears,drive_train,cylinders,cylinders_volume_cc,weight_kg,has_particle_filter,fuel_category,primary_fuel,electric_range_km,electric_range_city_km,fuel_cons_comb_l100_km,co2_emission_grper_km,fuel_cons_comb_l100_wltp_km,fuel_cons_electric_comb_l100_wltp_km,co2_emission_grper_wltp_km,is_used,is_new,is_preregistered,had_accident,has_full_service_history,non_smoking,nr_prev_owners,is_rental,envir_standard,original_market,offer_type,country_code,seller_is_dealer,seller_type,registration_year,low_price_flag,high_power_flag,very_high_km_flag,year_beyond_reference_flag,new_and_used_flag,inconsistent_price_net_flag,invalid_vin_flag,n_missing_fields,car_age,km_per_year,n_options,n_equipment_comfort,n_equipment_entertainment,n_equipment_extra,n_equipment_safety,log_price,ideal_price,price_residual,price_category,ideal_price_p10,ideal_price_p90
2402,94661,19400.0,True,False,bmw,i3,36214.0,Car,Sedan,4.0,4.0,White,Metallic,Cloth,Grey,135.0,184.0,Automatic,1.0,Rear Wheel Drive,0.0,0.0,1365.0,False,Electric,Electricity,NaN,NaN,5.7,NaN,NaN,NaN,NaN,True,False,False,False,False,False,1.0,False,unknown,Italy,U,IT,True,Dealer,2021.0,False,False,False,False,False,False,False,10,4.0,9053.5,8,4,1,2,1,9.873028,19692.127396,-0.014946,in_line,18670.17054,21036.692169



=== OUTPUT JSON DELLA VALUTAZIONE COMPLETA E REALE ===
{
    "asking_price": 19400.0,
    "ideal_price": 19692.13,
    "price_category": "in_line",
    "anomaly_score": -0.0002,
    "risk_category": "LOW",
    "top_3_anomalous_features": "gears = 1.00, mediana comparabili = 7.00 [SHAP: -0.72] | cylinders = 0.00, mediana comparabili = 0.00 [SHAP: -0.61] | cylinders_volume_cc = 0.00, mediana comparabili = 0.00 [SHAP: -0.45]"
}
